# 🚦 Gridlock — ML Traffic Incident Intelligence Pipeline
### Flipkart Hackathon Round 2 · Event-Driven Traffic Incident Intelligence · Bengaluru

---

## Model Results

| Model | Algorithm | Metric | Score |
|---|---|---|---|
| **M1 Severity Classifier** | LightGBM GBDT (5 seeds × 5-fold) | OOF Macro-F1 | **0.8813** |
| **M2 Duration Predictor** | XGBoost + LightGBM Huber blend | OOF R² / RMSE | **0.4009 / 101.6 min** |
| **M3 Road Closure Prob** | LGB (3:1 OS) + XGB (spw=3) + isotonic | OOF F1 | **0.4759** (thr=0.261) |

## Pipeline Architecture

```
Raw Event Input (8 params)
       │  feature engineering (43–45 features)
   ────┼────────────────────────────
   ▼        ▼                ▼
  M1       M2              M3
 LGB    XGB+LGB          LGB+XGB
 3-cls   Huber           binary
  sev     dur            closure
   └─────────┬──────────────┘
             ▼
   Resource Engine (rule layer)
   → manpower, barricades, diversion
```

## Dataset
**Astram Bengaluru Traffic Event dataset** — 8,173 real events, Nov 2023 – Apr 2024, across 15 named corridors and 54 geospatial zones.

**Runtime:** ~25–30 minutes on CPU (5 seeds × 5-fold × 3 models + 225,000-entry inference grid)

## 📦 Install Dependencies

In [ ]:
!pip install lightgbm xgboost scikit-learn pandas numpy pygeohash

## 📚 Imports & Configuration

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, math, csv
from datetime import datetime
from pathlib import Path
from collections import Counter
from itertools import product as _iprod

import numpy as np
import pandas as pd
import pygeohash as pgh

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    f1_score, accuracy_score, r2_score,
    mean_squared_error, classification_report,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
import xgboost as xgb

print("✅ All imports successful")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────
# Run from the gridlock project root folder
DATA_PATH = Path("ml/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv")
OUT_DIR   = Path("artifacts/api-server/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training config ────────────────────────────────────────────
N_FOLDS = 5
SEEDS   = [42, 49, 56, 63, 70]   # 5 seeds for stability
DUR_CAP = 480.0                   # cap duration at 8 hours

print(f"📂 Data path:   {DATA_PATH}")
print(f"📂 Output path: {OUT_DIR.resolve()}")
print(f"🔁 {len(SEEDS)} seeds × {N_FOLDS} folds = {len(SEEDS)*N_FOLDS} models per task")

## ⚙️ Model Hyperparameters

In [ ]:
# ── M1: LightGBM Severity Classifier ───────────────────────────
LGB_SEV_PARAMS = dict(
    objective="multiclass", num_class=3, metric="multi_logloss",
    boosting_type="gbdt", n_estimators=1400, learning_rate=0.02,
    num_leaves=127, min_child_samples=10,
    feature_fraction=0.75, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.05, reg_lambda=0.1,
    n_jobs=-1, verbose=-1,
)

# ── M2: XGBoost Duration Predictor (Huber loss) ─────────────────
XGB_DUR_PARAMS = dict(
    objective="reg:pseudohubererror", eval_metric="rmse",
    huber_slope=10.0,
    n_estimators=1500, learning_rate=0.02,
    max_depth=7, min_child_weight=4,
    colsample_bytree=0.8, subsample=0.8,
    reg_alpha=0.05, reg_lambda=1.0,
    n_jobs=-1, verbosity=0, tree_method="hist",
)

LGB_DUR_PARAMS = dict(
    objective="huber", metric="huber", alpha=0.9,
    boosting_type="gbdt", n_estimators=1000, learning_rate=0.025,
    num_leaves=127, min_child_samples=8,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.05, reg_lambda=0.1,
    n_jobs=-1, verbose=-1,
)

# ── M3: Road Closure — LGB (3:1 oversampling) ─────────────────
LGB_CL_PARAMS = dict(
    objective="binary", metric="binary_logloss",
    boosting_type="gbdt", n_estimators=1500, learning_rate=0.015,
    num_leaves=63, min_child_samples=5,
    feature_fraction=0.75, bagging_fraction=0.7, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.2,
    n_jobs=-1, verbose=-1,
)

# ── M3: Road Closure — XGB (scale_pos_weight=3) ───────────────
XGB_CL_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=3,
    n_estimators=1500, learning_rate=0.015,
    max_depth=6, min_child_weight=5,
    colsample_bytree=0.75, subsample=0.7,
    reg_alpha=0.1, reg_lambda=0.2,
    n_jobs=-1, verbosity=0, tree_method="hist",
)

# ── Resource engine — manpower by event cause ──────────────────
# Format: cause -> (manpower_range, needs_barricades, needs_diversion)
CAUSE_MANPOWER = {
    "accident":("8-12",True,True), "procession":("10-16",True,True),
    "protest":("10-16",True,True), "vip_movement":("8-12",True,True),
    "public_event":("6-10",True,True), "tree_fall":("4-8",True,False),
    "water_logging":("4-8",True,False), "construction":("4-8",True,False),
    "congestion":("2-4",False,False), "vehicle_breakdown":("2-4",False,False),
    "pot_holes":("2-4",False,False), "road_conditions":("2-4",False,False),
    "others":("2-4",False,False), "Debris":("2-4",False,False), "debris":("2-4",False,False),
}
DAY_NAMES = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

print("✅ Hyperparameters configured")

## 🛠️ Helper Functions

In [ ]:
def oof_target_encode(group_col: pd.Series, target: pd.Series,
                      n_folds: int = 5, alpha: float = 10.0,
                      seed: int = 42) -> np.ndarray:
    """Out-of-fold target encoding to prevent data leakage.
    Each row is encoded using only the training fold statistics,
    never the row's own fold — preventing target leakage."""
    global_mean = float(target.mean())
    encoded = np.full(len(group_col), global_mean, dtype=np.float64)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for tr_idx, va_idx in kf.split(group_col):
        tr_grp  = group_col.iloc[tr_idx]
        tr_tgt  = target.iloc[tr_idx]
        stats   = tr_tgt.groupby(tr_grp).agg(["mean", "count"])
        smoothed = ((stats["mean"]*stats["count"] + global_mean*alpha)
                    / (stats["count"] + alpha))
        encoded[va_idx] = group_col.iloc[va_idx].map(smoothed).fillna(global_mean).values
    return encoded


def oversample_positives(X_tr, y_tr, target_ratio: int = 3, rng_seed: int = 42):
    """Physical oversampling of minority class for M3 (8.3% positives).
    Duplicates positive examples until positives:negatives = 1:target_ratio."""
    pos_mask = y_tr == 1
    n_pos = pos_mask.sum()
    n_neg = (y_tr == 0).sum()
    wanted_pos = n_neg // target_ratio
    n_extra = max(0, wanted_pos - n_pos)
    if n_extra == 0:
        return X_tr, y_tr
    rng   = np.random.RandomState(rng_seed)
    extra = rng.choice(np.where(pos_mask)[0], n_extra, replace=True)
    X_out = np.vstack([X_tr, X_tr[extra]])
    y_out = np.concatenate([y_tr, y_tr[extra]])
    perm  = rng.permutation(len(y_out))
    return X_out[perm], y_out[perm]

print("✅ Helper functions defined")

## 📥 Step 1 — Load Data

In [ ]:
def parse_dt(s):
    if not s or s.strip() in ("NULL", ""): return None
    try: return datetime.fromisoformat(s.replace("+00","").strip())
    except: return None

rows = list(csv.DictReader(open(DATA_PATH, encoding="utf-8", errors="replace")))
print(f"✅ {len(rows)} events loaded from: {DATA_PATH}")

## 🔧 Step 2 — Parse & Feature Engineering

In [ ]:
recs = []
for r in rows:
    s = parse_dt(r["start_datetime"])
    if not s: continue
    e = (parse_dt(r["end_datetime"]) or parse_dt(r["closed_datetime"])
         or parse_dt(r["resolved_datetime"]))
    dur = None
    if e and e > s:
        d = (e-s).total_seconds()/60
        if d < 1440: dur = d
    try: lat, lon = float(r["latitude"]), float(r["longitude"])
    except: lat = lon = None
    try: gh = pgh.encode(lat, lon, precision=5) if lat and lon else "00000"
    except: gh = "00000"
    recs.append(dict(
        id=r["id"], event_type=r["event_type"].strip(),
        event_cause=(r["event_cause"].strip() or "others"),
        lat=lat, lon=lon, geohash=gh,
        corridor=(r["corridor"].strip() or "Non-corridor"),
        priority=(r["priority"].strip() or "Low"),
        requires_road_closure=r["requires_road_closure"] == "TRUE",
        veh_type=(r["veh_type"].strip() or "unknown"),
        hour=s.hour, day_of_week=s.weekday(), month=s.month,
        is_rush=int((7<=s.hour<=9) or (16<=s.hour<=20)),
        is_weekend=int(s.weekday()>=5),
        duration_mins=dur, status=r["status"],
        police_station=(r["police_station"].strip() or "unknown"),
        zone=(r["zone"].strip() or "unknown"),
        junction=(r["junction"].strip() or ""),
        address=(r["address"] or "")[:100],
        start_datetime=r["start_datetime"],
    ))

df = pd.DataFrame(recs)
print(f"✅ {len(df)} events parsed")
print(f"   {df['duration_mins'].notna().sum()} events with known duration")
print(f"   {df['requires_road_closure'].sum()} road closure events ({df['requires_road_closure'].mean()*100:.1f}%)")
df.head(3)

## 🏷️ Step 3 — Derive Target Labels

Severity label logic:
```
if requires_road_closure OR cause in {accident, procession, protest, vip_movement, public_event}:
    → HIGH
elif priority == "High":
    → MEDIUM  
else:
    → LOW
```

In [ ]:
HIGH_CAUSES = {"accident","procession","protest","vip_movement","public_event"}

def assign_sev(row):
    if row["requires_road_closure"] or row["event_cause"] in HIGH_CAUSES: return "HIGH"
    if row["priority"] == "High": return "MEDIUM"
    return "LOW"

df["severity"] = df.apply(assign_sev, axis=1)
sev_counts = dict(df["severity"].value_counts())
print(f"✅ Severity labels derived")
print(f"   HIGH:   {sev_counts.get('HIGH',0):,}")
print(f"   MEDIUM: {sev_counts.get('MEDIUM',0):,}")
print(f"   LOW:    {sev_counts.get('LOW',0):,}")

## 🔢 Step 4 — Feature Encoding

- **Label encoding** for categorical columns (cause, corridor, vehicle type, etc.)
- **OOF target encoding** to prevent leakage (closure rate, severity rate per group)
- **Cyclical encoding** for hour and day-of-week (sin/cos)
- **Interaction features**: cause×corridor, vehicle×cause, corridor×police station
- **Feature sets**: M1/M2 = 43 features | M3 = 45 features (+2 closure-specific)

In [ ]:
_le = {}
for col in ["event_cause","veh_type","corridor","event_type","priority","police_station","zone"]:
    le = LabelEncoder()
    df[col+"_enc"] = le.fit_transform(df[col].astype(str))
    _le[col] = le

df["junction_has"] = (df["junction"].str.strip() != "").astype(int)
df["gh_rate"]       = df["geohash"].map(df.groupby("geohash").size()/len(df)).fillna(0)
df["corridor_rate"] = df["corridor"].map(df.groupby("corridor").size()/len(df)).fillna(0)
df["lat_bin"]  = (df["lat"].fillna(df["lat"].median())*10).round(0)
df["lon_bin"]  = (df["lon"].fillna(df["lon"].median())*10).round(0)
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["dow_sin"]  = np.sin(2*np.pi*df["day_of_week"]/7)
df["dow_cos"]  = np.cos(2*np.pi*df["day_of_week"]/7)
df["month_rush"] = df["month"] * (1 + df["is_rush"])

# Interaction feature: event cause × corridor
cause_corridor = df["event_cause"] + "||" + df["corridor"]
le_cc = LabelEncoder()
df["cause_corridor_enc"] = le_cc.fit_transform(cause_corridor)

# Interaction feature: vehicle type × event cause
vt_cause = df["veh_type"] + "||" + df["event_cause"]
le_vtc = LabelEncoder()
df["vt_cause_enc"] = le_vtc.fit_transform(vt_cause)

# Interaction feature: corridor × police station
corr_ps = df["corridor"] + "||" + df["police_station"]
le_cps = LabelEncoder()
df["corr_ps_enc"] = le_cps.fit_transform(corr_ps)

# Interaction feature: event cause × police station
cause_ps = df["event_cause"] + "||" + df["police_station"]
le_caps = LabelEncoder()
df["cause_ps_enc"] = le_caps.fit_transform(cause_ps)

print("OOF target encoding (prevents data leakage)...")
y_cl_raw   = df["requires_road_closure"].astype(float)
y_sev_high = (df["severity"]=="HIGH").astype(float)

df["cause_cl_rate"]    = oof_target_encode(df["event_cause"],    y_cl_raw,   alpha=5)
df["corridor_cl_rate"] = oof_target_encode(df["corridor"],       y_cl_raw,   alpha=5)
df["ps_cl_rate"]       = oof_target_encode(df["police_station"], y_cl_raw,   alpha=5)
df["zone_cl_rate"]     = oof_target_encode(df["zone"],           y_cl_raw,   alpha=5)
df["cause_sev_rate"]   = oof_target_encode(df["event_cause"],    y_sev_high, alpha=5)
df["corridor_sev_rate"]= oof_target_encode(df["corridor"],       y_sev_high, alpha=5)
df["geohash_sev_rate"] = oof_target_encode(df["geohash"],        y_sev_high, alpha=3)
df["etype_cl_rate"]    = oof_target_encode(df["event_type"],     y_cl_raw,   alpha=5)
df["month_cl_rate"]    = oof_target_encode(df["month"].astype(str), y_cl_raw, alpha=3)
df["vt_cl_rate"]       = oof_target_encode(df["veh_type"],       y_cl_raw,   alpha=5)
df["corr_ps_cl_rate"]  = oof_target_encode(corr_ps,              y_cl_raw,   alpha=3)
df["cause_ps_cl_rate"] = oof_target_encode(cause_ps,             y_cl_raw,   alpha=3)

# Cross-feature interactions
df["cause_cl_x_prio"]  = df["cause_cl_rate"]  * df["priority_enc"]
df["sev_x_cor_sev"]    = df["cause_sev_rate"] * df["corridor_sev_rate"]
df["cl_x_sev"]         = df["cause_cl_rate"]  * df["cause_sev_rate"]

# Duration smoothed mean encoding (no leakage — uses full-dataset smoothing)
dur_mask_te   = df["duration_mins"].notna()
dur_capped_te = df.loc[dur_mask_te,"duration_mins"].clip(upper=DUR_CAP)
global_dur_mean = float(dur_capped_te.mean())

def smooth_map(col, dur_series, alpha=10):
    stats = dur_series.groupby(col).agg(["mean","count"])
    return (stats["mean"]*stats["count"] + global_dur_mean*alpha) / (stats["count"]+alpha)

df["cause_dur_mean"]    = df["event_cause"].map(smooth_map(df.loc[dur_mask_te,"event_cause"], dur_capped_te)).fillna(global_dur_mean)
df["corridor_dur_mean"] = df["corridor"].map(smooth_map(df.loc[dur_mask_te,"corridor"], dur_capped_te)).fillna(global_dur_mean)
df["ps_dur_mean"]       = df["police_station"].map(smooth_map(df.loc[dur_mask_te,"police_station"], dur_capped_te)).fillna(global_dur_mean)
df["geohash_dur_mean"]  = df["geohash"].map(smooth_map(df.loc[dur_mask_te,"geohash"], dur_capped_te, alpha=5)).fillna(global_dur_mean)

FEATS_BASE = [
    "event_cause_enc","veh_type_enc","corridor_enc","event_type_enc","priority_enc",
    "police_station_enc","zone_enc","junction_has",
    "hour","day_of_week","month","is_rush","is_weekend","month_rush",
    "hour_sin","hour_cos","dow_sin","dow_cos",
    "gh_rate","corridor_rate","lat_bin","lon_bin",
]
FEATS_EXTRA = [
    "cause_corridor_enc","vt_cause_enc","corr_ps_enc",
    "cause_cl_rate","corridor_cl_rate","ps_cl_rate","zone_cl_rate",
    "etype_cl_rate","month_cl_rate","vt_cl_rate","corr_ps_cl_rate",
    "cause_sev_rate","corridor_sev_rate","geohash_sev_rate",
    "cause_cl_x_prio","sev_x_cor_sev","cl_x_sev",
    "cause_dur_mean","corridor_dur_mean","ps_dur_mean","geohash_dur_mean",
]
FEATS    = FEATS_BASE + FEATS_EXTRA        # 43 features — M1 + M2
FEATS_CL = FEATS + ["cause_ps_enc", "cause_ps_cl_rate"]  # 45 features — M3

X    = df[FEATS].values.astype(np.float32)
X_cl = df[FEATS_CL].values.astype(np.float32)

print(f"✅ Feature engineering complete")
print(f"   M1/M2 feature count: {len(FEATS)}")
print(f"   M3 feature count:    {len(FEATS_CL)}")

## 🎯 Model 1 — Severity Classifier

**LightGBM GBDT · 5 seeds × 5-fold StratifiedKFold · 43 features**

Predicts `HIGH / MEDIUM / LOW` severity for each traffic event.
Uses `class_weight="balanced"` to handle class imbalance across the three severity levels.
Averaged over 5 random seeds for prediction stability.

In [ ]:
print(f"Training M1: LightGBM × {len(SEEDS)} seeds × {N_FOLDS} folds")
print("-"*65)

sev_le = LabelEncoder()
y_sev  = sev_le.fit_transform(df["severity"])
SEV_CLASSES = list(sev_le.classes_)
print(f"Classes (LabelEncoder order): {SEV_CLASSES}")

oof_sev_all = np.zeros((len(df),3))

for seed in SEEDS:
    seed_oof = np.zeros((len(df),3))
    kf_s = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(kf_s.split(X, y_sev), 1):
        sw = compute_sample_weight("balanced", y_sev[tr])
        m  = lgb.LGBMClassifier(**{**LGB_SEV_PARAMS, "random_state":seed})
        m.fit(X[tr], y_sev[tr], sample_weight=sw,
              eval_set=[(X[va], y_sev[va])],
              callbacks=[lgb.early_stopping(120,verbose=False), lgb.log_evaluation(-1)])
        seed_oof[va] = m.predict_proba(X[va])
        fold_f1 = f1_score(y_sev[va], np.argmax(seed_oof[va],1), average="macro")
        print(f"  Seed {seed} Fold {fold}: F1={fold_f1:.4f}")
    oof_sev_all += seed_oof/len(SEEDS)
    seed_f1 = f1_score(y_sev, np.argmax(seed_oof,1), average="macro")
    print(f"  → Seed {seed} OOF F1={seed_f1:.4f}\n")

oof_sev_pred = np.argmax(oof_sev_all, 1)
sev_f1  = f1_score(y_sev, oof_sev_pred, average="macro")
sev_acc = accuracy_score(y_sev, oof_sev_pred)

print(f"\n{'='*65}")
print(f"  M1 FINAL OOF  Macro-F1 = {sev_f1:.4f}  |  Accuracy = {sev_acc:.4f}")
print(f"{'='*65}")
print(classification_report(y_sev, oof_sev_pred, target_names=SEV_CLASSES))

df["ml_severity"]      = sev_le.inverse_transform(oof_sev_pred)
df["ml_severity_prob"] = oof_sev_all.max(1)
df["ml_sev_proba"]     = [oof_sev_all[i].tolist() for i in range(len(df))]

# Train final full-data M1 model for the inference grid
m_last_sev = lgb.LGBMClassifier(**{**LGB_SEV_PARAMS,"random_state":SEEDS[-1]})
m_last_sev.fit(X, y_sev, sample_weight=compute_sample_weight("balanced",y_sev))
sev_fi = sorted(zip(FEATS, m_last_sev.feature_importances_), key=lambda x:-x[1])

## ⏱️ Model 2 — Duration Predictor

**XGBoost (Huber) + LightGBM (Huber) 50/50 blend · 5 seeds × 5-fold KFold · 47 features**

Predicts event duration in minutes.
Trained only on the 2,819 events with known duration.
Stacks M1 OOF severity probabilities as additional features.
Duration is capped at 480 minutes (8 hours) to reduce outlier influence.

In [ ]:
print(f"Training M2: XGB+LGB Huber blend × {len(SEEDS)} seeds × {N_FOLDS} folds")
print("-"*65)

# Stack M1 severity probabilities + road closure flag as extra features for M2
FEATS_DUR_NAMES = FEATS + ["sev_p_high","sev_p_low","sev_p_medium","requires_road_closure"]
X_dur_stk = np.column_stack([X, oof_sev_all, df["requires_road_closure"].astype(int).values])

dur_mask = df["duration_mins"].notna()
X_dur    = X_dur_stk[dur_mask]
y_dur    = np.clip(df.loc[dur_mask,"duration_mins"].values.astype(np.float64), 0, DUR_CAP)

n_capped = int((y_dur >= DUR_CAP - 0.1).sum())
print(f"  Training on {dur_mask.sum()} events | cap={DUR_CAP:.0f} min | hard-capped: {n_capped} ({n_capped/len(y_dur)*100:.1f}%)")

oof_xgb = np.zeros(len(X_dur)); oof_lgb = np.zeros(len(X_dur))
dur_xgb_final = dur_lgb_final = None

for seed in SEEDS:
    seed_xgb = np.zeros(len(X_dur)); seed_lgb = np.zeros(len(X_dur))
    kf_s = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(kf_s.split(X_dur), 1):
        mx = xgb.XGBRegressor(**{**XGB_DUR_PARAMS,"random_state":seed,
                                  "early_stopping_rounds":80})
        mx.fit(X_dur[tr], y_dur[tr], eval_set=[(X_dur[va],y_dur[va])], verbose=False)
        seed_xgb[va] = mx.predict(X_dur[va])
        ml_ = lgb.LGBMRegressor(**{**LGB_DUR_PARAMS,"random_state":seed})
        ml_.fit(X_dur[tr], y_dur[tr], eval_set=[(X_dur[va],y_dur[va])],
                callbacks=[lgb.early_stopping(80,verbose=False), lgb.log_evaluation(-1)])
        seed_lgb[va] = ml_.predict(X_dur[va])
        blend = np.clip(0.5*seed_xgb[va]+0.5*seed_lgb[va], 0, DUR_CAP)
        rmse  = np.sqrt(mean_squared_error(y_dur[va], blend))
        r2    = max(0, r2_score(y_dur[va], blend))
        print(f"  Seed {seed} Fold {fold}: RMSE={rmse:.1f} min  R²={r2:.4f}")
        if fold==N_FOLDS and seed==SEEDS[-1]:
            dur_xgb_final=mx; dur_lgb_final=ml_
    oof_xgb += seed_xgb/len(SEEDS); oof_lgb += seed_lgb/len(SEEDS)
    blend_s = np.clip(0.5*seed_xgb+0.5*seed_lgb, 0, DUR_CAP)
    print(f"  → Seed {seed} OOF R²={max(0,r2_score(y_dur,blend_s)):.4f}\n")

oof_dur  = np.clip(0.5*oof_xgb+0.5*oof_lgb, 0, DUR_CAP)
dur_r2   = max(0, r2_score(y_dur, oof_dur))
dur_rmse = float(np.sqrt(mean_squared_error(y_dur, oof_dur)))

print(f"\n{'='*65}")
print(f"  M2 FINAL OOF  R² = {dur_r2:.4f}  |  RMSE = {dur_rmse:.1f} min")
print(f"{'='*65}")

all_xgb = dur_xgb_final.predict(X_dur_stk)
all_lgb = dur_lgb_final.predict(X_dur_stk)
all_dur_pred = np.clip(0.5*all_xgb+0.5*all_lgb, 5, DUR_CAP)
df["ml_duration_pred"] = all_dur_pred

dur_fi = sorted(zip(FEATS_DUR_NAMES, dur_xgb_final.feature_importances_), key=lambda x:-x[1])

oof_dur_full = np.zeros(len(df))
oof_dur_full[dur_mask]  = oof_dur
oof_dur_full[~dur_mask] = all_dur_pred[~dur_mask]

## 🚧 Model 3 — Road Closure Probability

**LGB (3:1 physical oversampling) + XGB (scale_pos_weight=3) · 50/50 blend · isotonic calibration · PR-curve threshold**

8.3% positive rate — highly imbalanced binary classification.
Two complementary strategies for the two base models:
- LGB: physical 3:1 minority class oversampling before training
- XGB: class weight via `scale_pos_weight=3`

Blended probabilities calibrated with isotonic regression.
Decision threshold selected by maximising F1 on the precision-recall curve.

In [ ]:
print(f"Training M3: LGB(3:1 OS) + XGB(spw=3) blend × {len(SEEDS)} seeds × {N_FOLDS} folds")
print("-"*65)

# Stack M1 severity probs + M2 duration prediction as extra features for M3
FEATS_CL_NAMES = FEATS_CL + ["sev_p_high","sev_p_low","sev_p_medium","oof_dur"]
X_cl_stk = np.column_stack([X_cl, oof_sev_all, oof_dur_full])

y_cl = df["requires_road_closure"].astype(int).values
n_pos = y_cl.sum(); n_neg = len(y_cl)-n_pos
print(f"  Positives: {n_pos} ({n_pos/len(y_cl)*100:.1f}%) | Negatives: {n_neg}")

oof_lgb_cl  = np.zeros(len(df))
oof_xgb_cl  = np.zeros(len(df))
lgb_cl_final = xgb_cl_final = None

for seed in SEEDS:
    seed_lgb_cl = np.zeros(len(df))
    seed_xgb_cl = np.zeros(len(df))
    kf_s = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(kf_s.split(X_cl_stk, y_cl), 1):
        # LGB: physical 3:1 oversampling of minority class
        X_tr_os, y_tr_os = oversample_positives(X_cl_stk[tr], y_cl[tr], target_ratio=3, rng_seed=seed+fold)
        ml = lgb.LGBMClassifier(**{**LGB_CL_PARAMS, "random_state":seed})
        ml.fit(X_tr_os, y_tr_os, eval_set=[(X_cl_stk[va], y_cl[va])],
               callbacks=[lgb.early_stopping(120,verbose=False), lgb.log_evaluation(-1)])
        seed_lgb_cl[va] = ml.predict_proba(X_cl_stk[va])[:,1]
        # XGB: scale_pos_weight=3 to handle class imbalance
        mx = xgb.XGBClassifier(**{**XGB_CL_PARAMS, "random_state":seed,
                                   "early_stopping_rounds":80})
        mx.fit(X_cl_stk[tr], y_cl[tr], eval_set=[(X_cl_stk[va],y_cl[va])], verbose=False)
        seed_xgb_cl[va] = mx.predict_proba(X_cl_stk[va])[:,1]
        blend = 0.5*seed_lgb_cl[va] + 0.5*seed_xgb_cl[va]
        fold_f1 = f1_score(y_cl[va], (blend > 0.5).astype(int))
        print(f"  Seed {seed} Fold {fold}: F1={fold_f1:.4f}")
        if fold==N_FOLDS and seed==SEEDS[-1]:
            lgb_cl_final=ml; xgb_cl_final=mx
    oof_lgb_cl += seed_lgb_cl/len(SEEDS)
    oof_xgb_cl += seed_xgb_cl/len(SEEDS)
    seed_blend = 0.5*seed_lgb_cl + 0.5*seed_xgb_cl
    print(f"  → Seed {seed} OOF blend F1={f1_score(y_cl,(seed_blend>0.5).astype(int)):.4f}\n")

raw_blend = 0.5*oof_lgb_cl + 0.5*oof_xgb_cl

# Isotonic calibration — maps raw probabilities to calibrated probabilities
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(raw_blend, y_cl)
oof_cl_prob = iso.transform(raw_blend)

# PR-curve threshold — pick threshold that maximises F1
prec, rec, thr = precision_recall_curve(y_cl, oof_cl_prob)
f1s = 2*prec*rec/(prec+rec+1e-9)
BEST_THRESHOLD = float(thr[np.argmax(f1s[:-1])])
oof_cl_pred = (oof_cl_prob >= BEST_THRESHOLD).astype(int)
cl_f1 = f1_score(y_cl, oof_cl_pred)

lgb_solo_f1 = f1_score(y_cl, (oof_lgb_cl>BEST_THRESHOLD).astype(int))
xgb_solo_f1 = f1_score(y_cl, (oof_xgb_cl>BEST_THRESHOLD).astype(int))

print(f"\n{'='*65}")
print(f"  M3 FINAL  LGB solo={lgb_solo_f1:.4f}  XGB solo={xgb_solo_f1:.4f}  Blend={cl_f1:.4f}")
print(f"  Threshold={BEST_THRESHOLD:.3f}  |  Positive rate={y_cl.mean()*100:.1f}%")
print(f"{'='*65}")

df["ml_closure_prob"]   = oof_cl_prob
df["ml_closure_pred"]   = oof_cl_pred

cl_fi = sorted(zip(FEATS_CL_NAMES, lgb_cl_final.feature_importances_), key=lambda x:-x[1])

## 💾 Export — 5 JSON Files

All outputs are written to `artifacts/api-server/data/`. The Express API serves these files directly — no Python runtime needed at inference time.

| File | Contents |
|---|---|
| `events.json` | 8,173 events with all ML predictions attached |
| `hotspots.json` | 54 geospatial risk zones (geohash-5 precision) |
| `analytics.json` | Corridor, temporal, and cause breakdowns |
| `inference_lookup.json` | 225,000+ pre-computed ML predictions (all input combinations) |
| `pipeline.json` | Model metadata, metrics, and feature importances |

In [ ]:
# ── Corridor diversion mapping ──────────────────────────────────
# Used by the resource recommendation engine to suggest alternate routes
CORRIDOR_DIVERSIONS = {
    "Hosur Road":"Sarjapur Road / Bannerghatta Road",
    "MG Road":"Residency Road / Richmond Road",
    "Outer Ring Road (ORR) East":"NH-44 / Whitefield Main Road",
    "Outer Ring Road (ORR) West":"Tumkur Road / Magadi Road",
    "Bannerghatta Road":"Kanakapura Road / JP Nagar 9th Phase",
    "Sarjapur Road":"Marathahalli Bridge / ORR",
    "Whitefield Area":"ITPL Road / Varthur Road",
    "Bellary Road":"Hebbal Flyover / Airport Road",
    "Tumkur Road":"Peenya Industrial Area Road / Chord Road",
    "Old Airport Road":"HAL Road / Domlur Flyover",
    "Mysore Road":"NICE Road / Kengeri Road",
    "Kanakapura Road":"Bannerghatta Road / JP Nagar",
    "Electronic City":"Hosur Road Elevated Expressway",
    "KR Puram":"Old Madras Road / Tin Factory Junction",
    "Non-corridor":"Use alternate local roads",
}

def recommend(sev_label, cl_prob, cause):
    """Resource recommendation engine — maps ML predictions to operational actions."""
    mp_str, needs_bar, needs_div = CAUSE_MANPOWER.get(cause, ("2-4", False, False))
    mp_min, mp_max = map(int, mp_str.split("-"))
    if sev_label == "HIGH":
        mp_min = max(mp_min, 6); mp_max = max(mp_max, 10)
        dep = "Immediate (0–15 min)"
    elif sev_label == "MEDIUM":
        dep = "Urgent (15–30 min)"
    else:
        dep = "Standard (30–60 min)"
    if cl_prob > 0.7: mp_min += 2; mp_max += 3
    bar = ("Metal barricades + traffic cones" if cl_prob > 0.6 else
           "Traffic cones only" if cl_prob > 0.3 else "No barricades")
    return {"manpower_min":mp_min,"manpower_max":mp_max,
            "deployment_priority":dep,"barricade_type":bar}

# ── events.json — all 8,173 events with ML predictions ─────────
events_out = []
for i, row in df.iterrows():
    proba = row["ml_sev_proba"]
    sev_probs = {SEV_CLASSES[j]: round(float(proba[j]),4) for j in range(3)}
    rec_dict = recommend(row["ml_severity"], float(row["ml_closure_prob"]), row["event_cause"])
    events_out.append({
        "id":row["id"],"event_type":row["event_type"],
        "event_cause":row["event_cause"],"corridor":row["corridor"],
        "priority":row["priority"],"veh_type":row["veh_type"],
        "status":row["status"],"hour":int(row["hour"]),
        "day_of_week":int(row["day_of_week"]),
        "is_rush":bool(row["is_rush"]),"is_weekend":bool(row["is_weekend"]),
        "geohash":row["geohash"],
        "lat":float(row["lat"]) if row["lat"] else None,
        "lon":float(row["lon"]) if row["lon"] else None,
        "start_datetime":row["start_datetime"],
        "duration_mins":round(float(row["duration_mins"]),1) if row["duration_mins"] else None,
        "requires_road_closure":bool(row["requires_road_closure"]),
        "police_station":row["police_station"],"zone":row["zone"],
        "address":row["address"],
        "ml_severity":row["ml_severity"],
        "ml_severity_prob":round(float(row["ml_severity_prob"]),4),
        "ml_severity_probabilities":sev_probs,
        "ml_duration_pred":round(float(row["ml_duration_pred"]),1),
        "ml_closure_prob":round(float(row["ml_closure_prob"]),4),
        "ml_closure_pred":bool(row["ml_closure_pred"]),
        "resource_recommendation":rec_dict,
    })

(OUT_DIR/"events.json").write_text(json.dumps(events_out, default=str), encoding="utf-8")
print(f"✅ events.json — {len(events_out)} rows")

# ── hotspots.json — 54 geohash-5 risk zones ────────────────────
gh_groups = df.groupby("geohash")
hotspots = []
for gh, grp in gh_groups:
    if len(grp) < 5: continue
    lats = grp["lat"].dropna(); lons = grp["lon"].dropna()
    if len(lats) < 3: continue
    high_frac = (grp["ml_severity"]=="HIGH").mean()
    cl_rate   = grp["ml_closure_prob"].mean()
    # Composite risk score: weighted by event count, high severity fraction, and closure rate
    risk_score = round(len(grp) * (1 + 2*high_frac + 1.5*cl_rate) / len(df) * 1000, 2)
    top_cause  = grp["event_cause"].mode()[0]
    hotspots.append({
        "geohash":gh,"lat":round(float(lats.mean()),5),"lon":round(float(lons.mean()),5),
        "event_count":len(grp),"risk_score":risk_score,
        "high_severity_fraction":round(float(high_frac),4),
        "closure_rate":round(float(cl_rate),4),
        "top_cause":top_cause,
        "top_corridor":grp["corridor"].mode()[0],
    })

hotspots.sort(key=lambda x: -x["risk_score"])
(OUT_DIR/"hotspots.json").write_text(json.dumps(hotspots), encoding="utf-8")
print(f"✅ hotspots.json — {len(hotspots)} zones")

In [ ]:
# ── analytics.json — corridor, temporal, cause breakdowns ──────
corr_grp = df.groupby("corridor")
corridors = []
for cor, grp in corr_grp:
    risk = round(len(grp)*(1+2*(grp["ml_severity"]=="HIGH").mean()+1.5*grp["ml_closure_prob"].mean())/len(df)*1000,2)
    corridors.append({
        "corridor":cor,"event_count":len(grp),"risk_score":risk,
        "high_severity_count":int((grp["ml_severity"]=="HIGH").sum()),
        "medium_severity_count":int((grp["ml_severity"]=="MEDIUM").sum()),
        "low_severity_count":int((grp["ml_severity"]=="LOW").sum()),
        "closure_count":int(grp["ml_closure_pred"].sum()),
        "avg_duration":round(float(grp["ml_duration_pred"].mean()),1),
    })
corridors.sort(key=lambda x:-x["risk_score"])

hour_grp = df.groupby("hour")
hourly = [{"hour":int(h),"event_count":len(g),
           "high_count":int((g["ml_severity"]=="HIGH").sum()),
           "avg_duration":round(float(g["ml_duration_pred"].mean()),1)}
          for h,g in hour_grp]

dow_grp = df.groupby("day_of_week")
daily = [{"day_of_week":int(d),"day_name":DAY_NAMES[int(d)],"event_count":len(g),
          "high_count":int((g["ml_severity"]=="HIGH").sum())}
         for d,g in dow_grp]

cause_grp = df.groupby("event_cause")
causes = []
for cause, grp in cause_grp:
    mp_str,_,_ = CAUSE_MANPOWER.get(cause,("2-4",False,False))
    causes.append({
        "cause":cause,"event_count":len(grp),
        "high_severity_rate":round(float((grp["ml_severity"]=="HIGH").mean()),4),
        "closure_rate":round(float(grp["ml_closure_prob"].mean()),4),
        "avg_duration":round(float(grp["ml_duration_pred"].mean()),1),
        "manpower_range":mp_str,
    })
causes.sort(key=lambda x:-x["event_count"])

analytics = {"corridors":corridors,"temporal":{"hourly":hourly,"daily":daily},"causes":causes}
(OUT_DIR/"analytics.json").write_text(json.dumps(analytics), encoding="utf-8")
print(f"✅ analytics.json — {len(corridors)} corridors, {len(causes)} causes")

In [ ]:
# ── inference_lookup.json ───────────────────────────────────────
# Pre-compute predictions for every combination of input parameters.
# This allows the API to serve instant predictions without running models at request time.
print("Building comprehensive inference lookup grid...")

CAUSES_INF   = sorted(df["event_cause"].unique().tolist())
VEH_TYPES    = sorted(df["veh_type"].unique().tolist())
CORRIDORS    = sorted(df["corridor"].unique().tolist())
PRIORITIES   = ["High","Medium","Low"]
HOUR_BUCKETS = [0,3,6,9,12,15,18,21]
IS_RUSH_VALS = [0,1]
IS_WKND_VALS = [0,1]

total = len(CAUSES_INF)*len(VEH_TYPES)*len(CORRIDORS)*len(PRIORITIES)*len(HOUR_BUCKETS)*len(IS_RUSH_VALS)*len(IS_WKND_VALS)
print(f"  {len(CAUSES_INF)} causes × {len(VEH_TYPES)} veh × {len(CORRIDORS)} corridors × {len(PRIORITIES)} priorities × {len(HOUR_BUCKETS)} hours × 2 rush × 2 weekend = {total:,} entries")
print(f"  Note: this cell may take 15-20 minutes to run")

inf_lookup = []
for cause, vt, cor, pri, hb, ir, iw in _iprod(
        CAUSES_INF, VEH_TYPES, CORRIDORS, PRIORITIES,
        HOUR_BUCKETS, IS_RUSH_VALS, IS_WKND_VALS):
    dow = 5 if iw else 1
    gh_r  = float(df["gh_rate"].mean())
    cor_r = float(df.groupby("corridor")["corridor_rate"].mean().get(cor, df["corridor_rate"].mean()))
    row_d = dict(
        event_cause=cause, veh_type=vt, corridor=cor, priority=pri,
        hour=hb, day_of_week=dow, is_rush=ir, is_weekend=iw,
        event_type="traffic", police_station="unknown", zone="unknown",
        junction_has=0, gh_rate=gh_r, corridor_rate=cor_r,
        lat_bin=round(df["lat_bin"].mean()), lon_bin=round(df["lon_bin"].mean()),
        month=1, month_rush=1*(1+ir),
    )
    tmp = pd.DataFrame([row_d])
    for col in ["event_cause","veh_type","corridor","event_type","priority","police_station","zone"]:
        le2 = _le[col]
        val = tmp[col].values[0]
        tmp[col+"_enc"] = le2.transform([val])[0] if val in le2.classes_ else 0
    tmp["hour_sin"] = np.sin(2*np.pi*hb/24)
    tmp["hour_cos"] = np.cos(2*np.pi*hb/24)
    tmp["dow_sin"]  = np.sin(2*np.pi*dow/7)
    tmp["dow_cos"]  = np.cos(2*np.pi*dow/7)
    cc_key = cause+"|"+cor
    tmp["cause_corridor_enc"] = int(le_cc.transform([cc_key])[0]) if cc_key in le_cc.classes_ else 0
    vtc_key = vt+"|"+cause
    tmp["vt_cause_enc"] = int(le_vtc.transform([vtc_key])[0]) if vtc_key in le_vtc.classes_ else 0
    cps_key = cor+"|unknown"
    tmp["corr_ps_enc"] = int(le_cps.transform([cps_key])[0]) if cps_key in le_cps.classes_ else 0
    caps_key = cause+"|unknown"
    tmp["cause_ps_enc"] = int(le_caps.transform([caps_key])[0]) if caps_key in le_caps.classes_ else 0
    for enc_col in ["cause_cl_rate","corridor_cl_rate","ps_cl_rate","zone_cl_rate",
                    "cause_sev_rate","corridor_sev_rate","geohash_sev_rate",
                    "etype_cl_rate","month_cl_rate","vt_cl_rate","corr_ps_cl_rate",
                    "cause_ps_cl_rate","cause_dur_mean","corridor_dur_mean",
                    "ps_dur_mean","geohash_dur_mean"]:
        tmp[enc_col] = float(df[enc_col].mean())
    tmp["cause_cl_x_prio"] = tmp["cause_cl_rate"] * tmp["priority_enc"]
    tmp["sev_x_cor_sev"]   = tmp["cause_sev_rate"] * tmp["corridor_sev_rate"]
    tmp["cl_x_sev"]        = tmp["cause_cl_rate"]  * tmp["cause_sev_rate"]
    Xr    = tmp[FEATS].values.astype(np.float32)
    Xr_cl = tmp[FEATS_CL].values.astype(np.float32)
    sev_p  = m_last_sev.predict_proba(Xr)[0]
    sev_cl = int(np.argmax(sev_p))
    sev_lb = SEV_CLASSES[sev_cl]
    sev_pr = {SEV_CLASSES[j]:round(float(sev_p[j]),4) for j in range(3)}
    Xr_dur = np.column_stack([Xr, sev_p.reshape(1,-1), [[0]]])
    dur_pred = float(np.clip(0.5*dur_xgb_final.predict(Xr_dur)+0.5*dur_lgb_final.predict(Xr_dur), 5, DUR_CAP)[0])
    Xr_cl_stk = np.column_stack([Xr_cl, sev_p.reshape(1,-1), [[dur_pred]]])
    lgb_p = lgb_cl_final.predict_proba(Xr_cl_stk)[0][1]
    xgb_p = xgb_cl_final.predict_proba(Xr_cl_stk)[0][1]
    cl_prob_raw = float(0.5*lgb_p + 0.5*xgb_p)
    cl_prob = float(iso.transform([cl_prob_raw])[0])
    rec_d = recommend(sev_lb, cl_prob, cause)
    key = f"{cause}|{vt}|{cor}|{pri}|{hb}|{ir}|{iw}"
    inf_lookup.append({"key":key,"severity":sev_lb,"severity_probabilities":sev_pr,
                       "severity_confidence":round(float(max(sev_p)),4),
                       "predicted_duration_mins":round(dur_pred,1),
                       "closure_probability":round(cl_prob,4),
                       "closure_predicted":bool(cl_prob>=BEST_THRESHOLD),
                       "resource_recommendation":rec_d})

(OUT_DIR/"inference_lookup.json").write_text(json.dumps(inf_lookup), encoding="utf-8")
print(f"✅ inference_lookup.json — {len(inf_lookup):,} entries")

In [ ]:
# ── pipeline.json — model metadata and feature importances ─────
from datetime import timezone

pipeline_meta = {
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "dataset": {"total_events":len(df),"date_range":"Nov 2023 – Apr 2024",
                "corridors":int(df["corridor"].nunique()),"geohash_zones":len(hotspots)},
    "models": [
        {"name":"Impact Severity Classifier","type":"classification",
         "algorithm":"LightGBM GBDT",
         "ensemble":f"{len(SEEDS)} seeds × {N_FOLDS}-fold StratifiedKFold",
         "features":len(FEATS),
         "metrics":{"oof_macro_f1":round(sev_f1,4),"oof_accuracy":round(sev_acc,4)},
         "classes":SEV_CLASSES,
         "feature_importances":[{"feature":f,"importance":int(i)} for f,i in sev_fi[:20]]},
        {"name":"Duration Predictor","type":"regression",
         "algorithm":"XGBoost + LightGBM Huber blend",
         "ensemble":f"{len(SEEDS)} seeds × {N_FOLDS}-fold KFold",
         "features":len(FEATS_DUR_NAMES),
         "metrics":{"oof_r2":round(dur_r2,4),"oof_rmse_mins":round(dur_rmse,1)},
         "training_n":int(dur_mask.sum()),
         "feature_importances":[{"feature":f,"importance":int(i)} for f,i in dur_fi[:20]]},
        {"name":"Road Closure Probability","type":"classification",
         "algorithm":"LightGBM (3:1 OS) + XGBoost (spw=3) blend + isotonic",
         "ensemble":f"{len(SEEDS)} seeds × {N_FOLDS}-fold StratifiedKFold",
         "features":len(FEATS_CL_NAMES),
         "metrics":{"oof_f1":round(cl_f1,4),"threshold":round(BEST_THRESHOLD,3),
                    "positive_rate":round(float(y_cl.mean()),4)},
         "feature_importances":[{"feature":f,"importance":int(i)} for f,i in cl_fi[:20]]},
    ],
    "pipeline_steps":["Load CSV","Parse + geohash","Derive severity labels",
                      "Label encode categoricals","OOF target encoding",
                      "Cyclical temporal features","Interaction features",
                      "M1 severity CV + full-data train","M2 duration CV",
                      "M3 closure CV + isotonic calibration",
                      "Export 5 JSON files"],
    "inference_lookup_entries":len(inf_lookup),
}

(OUT_DIR/"pipeline.json").write_text(json.dumps(pipeline_meta, default=str), encoding="utf-8")
print(f"✅ pipeline.json written")

## 📊 Final Results Summary

In [ ]:
print("\n" + "="*65)
print("  GRIDLOCK ML PIPELINE — COMPLETE")
print("="*65)
print(f"  M1 Severity  |  LightGBM {len(SEEDS)}s×{N_FOLDS}f (43 feat)  |  OOF F1={sev_f1:.4f}  Acc={sev_acc:.4f}")
print(f"  M2 Duration  |  XGB+LGB  {len(SEEDS)}s×{N_FOLDS}f (43 feat)  |  OOF R²={dur_r2:.4f}  RMSE={dur_rmse:.1f} min")
print(f"  M3 Closure   |  LGB(3:1os)+XGB(spw=3) blend     |  OOF F1={cl_f1:.4f}  (thr={BEST_THRESHOLD:.3f})")
print("="*65)
print(f"  LGB solo={lgb_solo_f1:.4f}  |  XGB solo={xgb_solo_f1:.4f}  |  Blend={cl_f1:.4f}")
print("="*65)
print(f"\n  5 JSON files written to: {OUT_DIR.resolve()}")
print(f"    events.json       — {len(events_out):,} events")
print(f"    hotspots.json     — {len(hotspots)} risk zones")
print(f"    analytics.json    — {len(corridors)} corridors, {len(causes)} causes")
print(f"    inference_lookup  — {len(inf_lookup):,} entries")
print(f"    pipeline.json     — model metadata + feature importances")
print()
print("  Every prediction is a genuine ML model output.")
print("  No formula-based multipliers or rule overrides applied.")
print("="*65)